# KANFuse: Multimodal Band-Gap Prediction with KAN Regression

Architecture:
1. **MatSciBERT** (frozen) → full token sequence → `TextTokenFinegrain` (self-attn + pool) → `[batch, PROJ_DIM]`
2. **ALIGNN** (from scratch) → direct readout → `graph_proj` → `[batch, PROJ_DIM]`
3. Concatenate → `LayerNorm` → `[batch, 2*PROJ_DIM]`
4. `KANLinear(2P → P)` → `KANLinear(P → P//4)` (B-spline, grid_range=(-4,4))
5. Linear regression head → band gap prediction

Key design choices:
- MatSciBERT weights frozen; only `TextTokenFinegrain` trains on text side
- ALIGNN trained from scratch jointly with the fusion head
- LayerNorm before KAN keeps inputs inside the B-spline grid range
- Single `AdamW(lr=1e-4)` for all trainable params
- `ReduceLROnPlateau(factor=0.5, patience=2)` stepping on val loss

In [ ]:
!pip uninstall -y torchao -q
!pip install -q torch==2.2.1 torchdata==0.7.1 'numpy<2' --index-url https://download.pytorch.org/whl/cu121
!pip install -q dgl -f https://data.dgl.ai/wheels/cu121/repo.html
!pip install -q 'transformers<5' alignn huggingface_hub pandas scikit-learn tqdm matplotlib
!pip uninstall -y torchao -q

In [ ]:
import os, json, random, hashlib, math, datetime
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
import dgl
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import AutoModel, AutoTokenizer
from alignn.models.alignn import ALIGNN, ALIGNNConfig
from huggingface_hub import snapshot_download, hf_hub_download
from sklearn.metrics import mean_absolute_error, r2_score
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
REPO_ID          = 'Godseye1311/alignn-band-gap'
LOCAL_DIR        = './kanfuse_data'
TRANSFORMER_NAME = 'm3rg-iitd/matscibert'

# Encoder split — keep for fair comparison with matMMFuse
ENCODER_SPLIT_SEED = 0
VAL_FRACTION       = 0.1
TEST_FRACTION      = 0.1

# Model hyper-parameters
HIDDEN_FEATURES = 256   # ALIGNN hidden dim
PROJ_DIM        = 256   # both modalities projected to this before concat
KAN_DIM1        = 256   # KAN layer 1 output
KAN_DIM2        = 64    # KAN layer 2 output
GRID_SIZE       = 5
SPLINE_ORD      = 3
ATTN_HEADS      = 4

BATCH_SIZE      = 64    # plateau-best recipe (kanfuse_full run); lower to 32 if Colab OOMs
BUFFER_SHARDS   = 12
NUM_EPOCHS      = 150
EARLY_STOP_PAT  = 15    # plateau-best recipe: give ReduceLROnPlateau room past LR drops
LR              = 1e-4
WEIGHT_DECAY    = 0.01
NORMALIZE_TARGETS = False

## Data Download

In [ ]:
local_path = snapshot_download(
    repo_id=REPO_ID, repo_type='dataset',
    local_dir=LOCAL_DIR, ignore_patterns=['xrd_data.h5'],
)
print('Downloaded to:', local_path)

In [ ]:
ID_COL, TEXT_COL, TARGET_COL = 'material_id', 'description', 'band_gap'

text_df    = pd.read_csv(os.path.join(LOCAL_DIR, 'text_data.csv'))
tabular_df = pd.read_csv(os.path.join(LOCAL_DIR, 'materials_tabular.csv'))

with open(os.path.join(LOCAL_DIR, 'alignn_graphs', 'graph_shards.json')) as f:
    graph_index = json.load(f)
print(f'Graphs available: {len(graph_index)}')

merged = text_df[[ID_COL, TEXT_COL]].merge(
    tabular_df[[ID_COL, TARGET_COL]], on=ID_COL, how='inner'
)
merged = merged.dropna(subset=[TEXT_COL, TARGET_COL])
merged = merged[merged[ID_COL].astype(str).isin(graph_index.keys())]
merged = merged.drop_duplicates(subset=[ID_COL]).reset_index(drop=True)
print(f'Usable rows: {len(merged)}')

In [ ]:
_enc_df  = tabular_df[tabular_df[ID_COL].isin(graph_index.keys())].reset_index(drop=True)
_enc_ids = _enc_df[ID_COL].astype(str).tolist()
_n       = len(_enc_ids)
_perm    = np.random.RandomState(ENCODER_SPLIT_SEED).permutation(_n)
_n_val   = int(VAL_FRACTION  * _n)
_n_test  = int(TEST_FRACTION * _n)
_n_train = _n - _n_val - _n_test

split_of = {}
for positions, name in (
    (_perm[:_n_train],                   'train'),
    (_perm[_n_train:_n_train + _n_val],  'val'),
    (_perm[_n_train + _n_val:],          'test'),
):
    for i in positions:
        split_of[_enc_ids[i]] = name

_split_file = os.path.join(LOCAL_DIR, 'alignn_split.json')
if os.path.exists(_split_file):
    with open(_split_file) as f:
        assert json.load(f) == split_of, 'split mismatch'
    print('Split matches alignn_split.json')

_sig = hashlib.sha1(
    '|'.join(f'{m}:{split_of[m]}' for m in sorted(split_of)).encode()
).hexdigest()[:12]
print(f'Split signature: {_sig}')

merged['split'] = merged[ID_COL].astype(str).map(split_of)
assert merged['split'].notna().all()
train_df = merged[merged['split'] == 'train'].reset_index(drop=True)
val_df   = merged[merged['split'] == 'val'].reset_index(drop=True)
test_df  = merged[merged['split'] == 'test'].reset_index(drop=True)
print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

## Dataset & DataLoaders

In [ ]:
MAX_CACHED_TRAIN = BUFFER_SHARDS + 2
MAX_CACHED_EVAL  = 4
_cache_train = OrderedDict()
_cache_eval  = OrderedDict()

def _load_shard(name, cache, max_size):
    if name in cache:
        cache.move_to_end(name)
        return cache[name]
    ag, _ = dgl.load_graphs(os.path.join(LOCAL_DIR, 'alignn_graphs', f'{name}_atom.bin'))
    lg, _ = dgl.load_graphs(os.path.join(LOCAL_DIR, 'alignn_graphs', f'{name}_line.bin'))
    cache[name] = (ag, lg)
    if len(cache) > max_size:
        cache.popitem(last=False)
    return cache[name]

def get_graph_pair(mid, is_eval=False):
    e = graph_index[str(mid)]
    ag, lg = _load_shard(e['shard'], _cache_eval if is_eval else _cache_train,
                         MAX_CACHED_EVAL if is_eval else MAX_CACHED_TRAIN)
    return ag[e['index']], lg[e['index']]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_NAME)

class ALIGNNTextDataset(Dataset):
    def __init__(self, df, is_eval=False):
        self.ids    = df[ID_COL].astype(str).tolist()
        self.texts  = df[TEXT_COL].tolist()
        self.labels = df[TARGET_COL].astype(float).tolist()
        self.is_eval = is_eval
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        g, lg = get_graph_pair(self.ids[idx], self.is_eval)
        return self.texts[idx], (g, lg), self.labels[idx]

def collate_fn(batch):
    texts, pairs, labels = zip(*batch)
    gs, lgs = zip(*pairs)
    return list(texts), (dgl.batch(gs), dgl.batch(lgs)), torch.tensor(labels, dtype=torch.float32)

class ShardBufferedSampler(Sampler):
    def __init__(self, dataset, buffer_shards, shuffle=True, seed=SEED):
        self.shard_to_indices = {}
        for i, mid in enumerate(dataset.ids):
            self.shard_to_indices.setdefault(graph_index[mid]['shard'], []).append(i)
        self.buffer_shards = buffer_shards
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0
    def set_epoch(self, e): self.epoch = e
    def __iter__(self):
        names = sorted(self.shard_to_indices)
        rng = random.Random(self.seed + self.epoch)
        if self.shuffle: rng.shuffle(names)
        for start in range(0, len(names), self.buffer_shards):
            block = names[start:start + self.buffer_shards]
            idxs = [i for s in block for i in self.shard_to_indices[s]]
            if self.shuffle: rng.shuffle(idxs)
            yield from idxs
    def __len__(self): return sum(len(v) for v in self.shard_to_indices.values())

train_ds = ALIGNNTextDataset(train_df, is_eval=False)
val_ds   = ALIGNNTextDataset(val_df,   is_eval=True)
test_ds  = ALIGNNTextDataset(test_df,  is_eval=True)

train_sampler = ShardBufferedSampler(train_ds, BUFFER_SHARDS, shuffle=True)
val_sampler   = ShardBufferedSampler(val_ds,   BUFFER_SHARDS, shuffle=False)
test_sampler  = ShardBufferedSampler(test_ds,  BUFFER_SHARDS, shuffle=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True,
                          persistent_workers=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, sampler=val_sampler,
                          collate_fn=collate_fn, num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, sampler=test_sampler,
                          collate_fn=collate_fn, num_workers=0)
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')

## ALIGNN Encoder — Direct Readout (from scratch)

`ALIGNNDirect` returns the pooled graph embedding directly from the readout layer
(before the fc head), giving `[batch, hidden_features]`. ALIGNN is trained from scratch
jointly with the rest of the model — no pretrained checkpoint is loaded.

In [ ]:
_g0, _ = get_graph_pair(merged[ID_COL].iloc[0], is_eval=True)
ATOM_INPUT_FEATURES = int(_g0.ndata['atom_features'].shape[-1])
print(f'atom_input_features={ATOM_INPUT_FEATURES}, hidden_features={HIDDEN_FEATURES}')

class ALIGNNDirect(ALIGNN):
    """ALIGNN returning the pooled graph embedding from readout (before fc layer)."""
    def forward(self, g_input):
        if len(g_input) == 3:
            g, lg, _ = g_input
        else:
            g, lg = g_input
        lg = lg.local_var()
        z  = self.angle_embedding(lg.edata.pop('h'))
        g  = g.local_var()
        x  = self.atom_embedding(g.ndata.pop('atom_features'))
        y  = self.edge_embedding(torch.norm(g.edata.pop('r'), dim=1))
        for layer in self.alignn_layers:
            x, y, z = layer(g, lg, x, y, z)
        for layer in self.gcn_layers:
            x, y = layer(g, x, y)
        return self.readout(g, x)  # [batch, hidden_features]

alignn_config = ALIGNNConfig(
    name='alignn',
    atom_input_features=ATOM_INPUT_FEATURES,
    hidden_features=HIDDEN_FEATURES,
    output_features=1,
    alignn_layers=4,
    gcn_layers=4,
    link='identity',
    classification=False,
)
alignn_model = ALIGNNDirect(alignn_config)
print('ALIGNN initialised from scratch (random weights)')

## KAN Linear Layer

B-spline KAN layer with `grid_range=(-4, 4)` — wider than the default (-1,1) so the
spline basis covers the full dynamic range of LayerNorm'd embeddings.

In [ ]:
class KANLinear(nn.Module):
    def __init__(self, in_features, out_features, grid_size=5, spline_order=3,
                 scale_noise=0.1, scale_base=1.0, scale_spline=1.0,
                 grid_eps=0.02, grid_range=(-4, 4)):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.grid_size    = grid_size
        self.spline_order = spline_order
        h    = (grid_range[1] - grid_range[0]) / grid_size
        grid = (
            torch.arange(-spline_order, grid_size + spline_order + 1) * h + grid_range[0]
        ).expand(in_features, -1).contiguous()
        self.register_buffer('grid', grid)
        self.base_weight   = nn.Parameter(torch.empty(out_features, in_features))
        self.spline_weight = nn.Parameter(
            torch.empty(out_features, in_features, grid_size + spline_order))
        self.spline_scaler = nn.Parameter(torch.empty(out_features, in_features))
        self.base_act      = nn.SiLU()
        self._reset()

    def _reset(self):
        nn.init.kaiming_uniform_(self.base_weight, a=math.sqrt(5))
        with torch.no_grad():
            noise = ((torch.rand(self.grid_size + 1, self.in_features, self.out_features)
                      - 0.5) * 0.1 / self.grid_size)
            self.spline_weight.data.copy_(self._c2c(
                self.grid.T[self.spline_order:-self.spline_order], noise))
        nn.init.kaiming_uniform_(self.spline_scaler, a=math.sqrt(5))

    def _bsplines(self, x):
        x = x.unsqueeze(-1)
        g = self.grid
        b = ((x >= g[:, :-1]) & (x < g[:, 1:])).to(x.dtype)
        for k in range(1, self.spline_order + 1):
            b = ((x - g[:, :-(k+1)]) / (g[:, k:-1] - g[:, :-(k+1)]) * b[:, :, :-1]
                 + (g[:, k+1:] - x) / (g[:, k+1:] - g[:, 1:-k]) * b[:, :, 1:])
        return b

    def _c2c(self, x, y):
        A   = self._bsplines(x).transpose(0, 1)
        B   = y.transpose(0, 1)
        sol = torch.linalg.lstsq(A, B).solution
        return sol.permute(2, 0, 1).contiguous()

    def forward(self, x):
        orig = x.shape
        x    = x.reshape(-1, self.in_features)
        sw   = self.spline_weight * self.spline_scaler.unsqueeze(-1)
        out  = (F.linear(self.base_act(x), self.base_weight)
                + F.linear(self._bsplines(x).view(x.shape[0], -1),
                           sw.view(self.out_features, -1)))
        return out.reshape(*orig[:-1], self.out_features)

## Token-Level Fine-Graining (Text)

`TextTokenFinegrain` applies one Transformer encoder layer across MatSciBERT's full
token sequence, then compresses to `[batch, proj_dim]` via a learned attention-pool query.

In [ ]:
class TextTokenFinegrain(nn.Module):
    def __init__(self, input_dim, proj_dim, num_heads=4, dropout=0.1):
        super().__init__()
        self.proj    = nn.Linear(input_dim, proj_dim)
        self.norm_in = nn.LayerNorm(proj_dim)
        enc = nn.TransformerEncoderLayer(
            d_model=proj_dim, nhead=num_heads, dim_feedforward=proj_dim * 2,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc, num_layers=1)
        self.pool_query  = nn.Parameter(torch.randn(1, 1, proj_dim) * 0.02)
        self.pool_attn   = nn.MultiheadAttention(proj_dim, num_heads, dropout=dropout,
                                                  batch_first=True)
        self.norm_out = nn.LayerNorm(proj_dim)

    def forward(self, x, attention_mask=None):
        x   = self.norm_in(self.proj(x))
        kpm = (attention_mask == 0) if attention_mask is not None else None
        x   = self.transformer(x, src_key_padding_mask=kpm)
        q   = self.pool_query.expand(x.shape[0], -1, -1)
        agg, _ = self.pool_attn(q, x, x, key_padding_mask=kpm)
        return self.norm_out(agg.squeeze(1))

## KANFuse Model

```
MatSciBERT (frozen) → last_hidden_state [batch, seq_len, 768]
                    → TextTokenFinegrain → text_emb  [batch, PROJ_DIM]

ALIGNN (scratch)    → readout            → graph_emb [batch, 256]
                    → graph_proj         → graph_emb [batch, PROJ_DIM]

cat([text_emb, graph_emb]) → LayerNorm  → [batch, 2*PROJ_DIM]
  → KANLinear(2P → P)  → KANLinear(P → P//4)  → Linear → band gap
```

In [ ]:
class KANFuseModel(nn.Module):
    def __init__(self, transformer_name, alignn_encoder,
                 graph_hidden=HIDDEN_FEATURES, proj_dim=PROJ_DIM,
                 kan_dim1=KAN_DIM1, kan_dim2=KAN_DIM2,
                 grid_size=GRID_SIZE, spline_order=SPLINE_ORD,
                 attn_heads=ATTN_HEADS, dropout=0.1):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(transformer_name)
        for p in self.transformer.parameters():
            p.requires_grad = False
        self.alignn         = alignn_encoder
        text_hidden         = self.transformer.config.hidden_size  # 768
        self.text_finegrain = TextTokenFinegrain(text_hidden, proj_dim, attn_heads, dropout)
        self.graph_proj = nn.Sequential(
            nn.Linear(graph_hidden, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.pre_kan_norm = nn.LayerNorm(2 * proj_dim)
        self.kan1 = KANLinear(2 * proj_dim, kan_dim1, grid_size=grid_size,
                              spline_order=spline_order)
        self.kan2 = KANLinear(kan_dim1, kan_dim2, grid_size=grid_size,
                              spline_order=spline_order)
        self.head = nn.Linear(kan_dim2, 1)

    def forward(self, texts, graph_pair):
        g, lg = graph_pair
        g, lg = g.to(device), lg.to(device)
        # Graph branch — fp32 forced (ALIGNN BatchNorm1d unstable under fp16)
        with torch.autocast(device_type='cuda', enabled=False):
            graph_emb = self.alignn((g, lg))              # [batch, hidden_features]
        graph_emb = self.graph_proj(graph_emb.float())    # [batch, proj_dim]
        # Text branch — frozen MatSciBERT, no grad
        inputs = tokenizer(texts, return_tensors='pt', padding=True,
                           truncation=True, max_length=512).to(device)
        with torch.no_grad():
            token_seq = self.transformer(**inputs).last_hidden_state  # [batch, seq_len, 768]
        text_emb = self.text_finegrain(token_seq, inputs['attention_mask'])  # [batch, proj_dim]
        fused = self.pre_kan_norm(torch.cat([text_emb, graph_emb], dim=1))
        return self.head(self.kan2(self.kan1(fused)))

model = KANFuseModel(TRANSFORMER_NAME, alignn_model).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

## Training

In [ ]:
os.makedirs('./Results/Checkpoint', exist_ok=True)

criterion   = nn.L1Loss()
TARGET_MEAN = float(train_df[TARGET_COL].mean()) if NORMALIZE_TARGETS else 0.0
TARGET_STD  = float(train_df[TARGET_COL].std())  if NORMALIZE_TARGETS else 1.0
print(f'target normalisation: {NORMALIZE_TARGETS}  (mean={TARGET_MEAN:.4f}, std={TARGET_STD:.4f})')

# Single LR for all trainable params (transformer.* frozen so auto-excluded)
optimizer = AdamW([p for p in model.parameters() if p.requires_grad],
                  lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

def evaluate(loader, desc='Val', amp=True):
    model.eval()
    total_loss, n = 0.0, 0
    preds_all, labels_all = [], []
    with torch.no_grad():
        for texts, gp, labels in tqdm(loader, desc=desc, leave=False):
            labels  = labels.to(device)
            targets = (labels - TARGET_MEAN) / TARGET_STD
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=amp):
                p = model(texts, gp).reshape(-1)
            p = p.float()
            total_loss += criterion(p, targets).item()
            n          += 1
            preds_all.extend((p * TARGET_STD + TARGET_MEAN).cpu().tolist())
            labels_all.extend(labels.cpu().tolist())
    model.train()
    return (total_loss / max(n, 1),
            mean_absolute_error(labels_all, preds_all),
            r2_score(labels_all, preds_all))

In [ ]:
best_val_mae           = float('inf')
best_epoch             = -1
epochs_no_improve      = 0
history                = []
model.train()

for epoch in range(NUM_EPOCHS):
    train_sampler.set_epoch(epoch)
    total_loss, nan_batches, seen = 0.0, 0, 0
    bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}', unit='batch')

    for texts, gp, labels in bar:
        labels  = labels.to(device)
        targets = (labels - TARGET_MEAN) / TARGET_STD
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            p = model(texts, gp).reshape(-1)
        loss = criterion(p.float(), targets)
        optimizer.zero_grad(set_to_none=True)
        if torch.isnan(loss):
            nan_batches += 1
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        seen       += 1
        bar.set_postfix(loss=f'{loss.item():.4f}', avg=f'{total_loss/max(seen,1):.4f}')

    train_avg = total_loss / max(seen, 1)
    val_loss, val_mae, val_r2 = evaluate(val_loader)
    scheduler.step(val_loss)
    history.append({'epoch': epoch+1, 'train_loss': train_avg,
                    'val_loss': val_loss, 'val_mae': val_mae, 'val_r2': val_r2})
    print(f'Epoch {epoch+1}: train={train_avg:.4f}  val={val_loss:.4f}  '
          f'val_MAE={val_mae:.4f}  val_R2={val_r2:.4f}  nan={nan_batches}')

    # Periodic checkpoint — trainable params only (transformer.* frozen, excluded)
    if (epoch + 1) % 5 == 0:
        ts = datetime.datetime.now().strftime('%Y%m%d-%H%M')
        trainable_state = {k: v for k, v in model.state_dict().items()
                           if not k.startswith('transformer.')}
        torch.save({'model_state_dict': trainable_state, 'epoch': epoch,
                    'train_loss': train_avg},
                   f'./Results/Checkpoint/kanfuse_epoch{epoch+1}_{ts}.pth')

    if val_mae < best_val_mae:
        best_val_mae, best_epoch = val_mae, epoch
        epochs_no_improve        = 0
        trainable_state = {k: v for k, v in model.state_dict().items()
                           if not k.startswith('transformer.')}
        torch.save({
            'model_state_dict':     trainable_state,
            'optimizer_state_dict': optimizer.state_dict(),
            'epoch':                epoch,
            'val_mae':              val_mae,
            'split_signature':      _sig,
            'target_mean':          TARGET_MEAN,
            'target_std':           TARGET_STD,
        }, './Results/Checkpoint/kanfuse_best.pth')
        print(f'  New best val MAE {best_val_mae:.4f} — checkpoint saved')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PAT:
            print(f'Early stop: no improvement for {EARLY_STOP_PAT} epochs '
                  f'(best {best_val_mae:.4f} @ epoch {best_epoch+1})')
            break

pd.DataFrame(history).to_csv('./Results/kanfuse_history.csv', index=False)
print(f'Best val MAE {best_val_mae:.4f} at epoch {best_epoch+1}')

## Test Evaluation

In [ ]:
ckpt = torch.load('./Results/Checkpoint/kanfuse_best.pth', map_location=device)
# strict=False: transformer.* excluded from checkpoint (frozen, reload from HF instead)
model.load_state_dict(ckpt['model_state_dict'], strict=False)
assert ckpt['split_signature'] == _sig
print(f"Loaded best checkpoint: epoch {ckpt['epoch']+1}, val MAE {ckpt['val_mae']:.4f}")

test_loss, test_mae, test_r2 = evaluate(test_loader, desc='Testing', amp=False)
print(f'Test L1 Loss : {test_loss:.4f}')
print(f'Test MAE     : {test_mae:.4f}   (MatMMFuse baseline: ~0.31 eV)')
print(f'Test R2      : {test_r2:.4f}')

In [ ]:
import matplotlib.pyplot as plt

hist_df = pd.read_csv('./Results/kanfuse_history.csv')
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hist_df['epoch'], hist_df['val_mae'], label='Val MAE', color='tab:blue', lw=1, alpha=0.6)
best_row = hist_df.loc[hist_df['val_mae'].idxmin()]
ax.scatter([best_row['epoch']], [best_row['val_mae']], color='red', zorder=5)
ax.annotate(f"min {best_row['val_mae']:.4f} @ ep {int(best_row['epoch'])}",
            (best_row['epoch'], best_row['val_mae']),
            textcoords='offset points', xytext=(8, -14), fontsize=9)
ax.set_xlabel('Epoch')
ax.set_ylabel('MAE (eV)')
ax.set_title('KANFuse — Validation MAE')
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.savefig('./Results/kanfuse_val_mae.png', dpi=150)
plt.show()

## Architecture Notes

| | MatMMFuse | Band_KAN_Fusion | KANFuse (this) |
|---|---|---|---|
| MatSciBERT | Fully trainable, mean-pool | Fully trainable, masked mean-pool | **Frozen**, token-level fine-grain |
| ALIGNN | Pretrained encoder | From scratch | **From scratch** |
| Graph rep | Pooled → 512-dim | Direct readout → 128-dim | Direct readout → 256-dim |
| Fusion | Cross-attention (degenerate) | KANLinear(256→256) | **LayerNorm → KANLinear(512→256→64)** |
| KAN grid | — | (-4, 4) | **(-4, 4)** |
| Loss | SmoothL1 | L1 | **L1** |
| Scheduler | Cosine warmup | ReduceLROnPlateau | **ReduceLROnPlateau** |
| LR | Per-branch | Single 1e-4 | **Single 1e-4** |
| Checkpoint | Full model | Full model | **Trainable params only (~25MB)** |